# 03 — ToN-IoT Data Preparation

**Goal:** Prepare reproducible training (70%), validation (15%) and testing (15%) partitions without removing rows.

**Input:** Core10 dataset from Notebook 02.

**Output:** Frozen split manifest, three CSV.GZ files, three Excel copies, one report and one graph.

**Rule:** Identical normalized feature patterns stay in the same partition. The test partition is not used for model selection.

In [ ]:
from pathlib import Path
import hashlib, json, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path(r'D:\J26-CS-336-Network-Flow-V2\network_flow_v2')
SELECTED = ROOT / 'data/processed/ton_iot/ton_iot_core10_selected_v3.csv.gz'
AUDIT1 = ROOT / 'reports/audits/ton_iot_v3_dataset_analysis.json'
AUDIT2 = ROOT / 'reports/audits/ton_iot_v3_feature_quality.json'
SPLITS = ROOT / 'data/splits/ton_iot'
MANIFEST = SPLITS / 'ton_iot_core10_signature_split_manifest_v3.csv'
REPORT = ROOT / 'reports/audits/ton_iot_v3_split_report.json'
FIGURE = ROOT / 'reports/figures/ton_iot_v3_split_class_balance.png'
EXCEL_SCRIPT = ROOT / 'scripts/export_ton_iot_splits_excel.ps1'
PARTS = ('train', 'validation', 'test')
FRACTIONS = np.array([0.70, 0.15, 0.15])
SEED = 42
EXPECTED_ROWS = 211043
EXPORT_EXCEL = True  # Microsoft Excel desktop is needed on Windows.
MISSING = {'', '-', '?', 'na', 'n/a', 'nan', 'none', 'null', 'missing', 'unknown'}

for path in (SELECTED, AUDIT1, AUDIT2):
    if not path.is_file():
        raise FileNotFoundError(f'Missing required file: {path}. Complete Notebook 01 and 02 first.')
for d in (SPLITS, REPORT.parent, FIGURE.parent):
    d.mkdir(parents=True, exist_ok=True)
a1 = json.loads(AUDIT1.read_text(encoding='utf-8'))
a2 = json.loads(AUDIT2.read_text(encoding='utf-8'))
FEATURES = a2['selected_features']
CATEGORICAL = a2['categorical_features']
NUMERIC = [name for name in FEATURES if name not in CATEGORICAL]
FIELDS = ['csv_row_index_0based'] + FEATURES + ['label', 'type']
if len(FEATURES) != 10 or len(NUMERIC) != 7 or len(CATEGORICAL) != 3:
    raise ValueError('Unexpected Core10 features. Check Notebook 02.')
if a1['rows'] != a2['rows'] or a2['rows'] != EXPECTED_ROWS or a2['exported_columns'] != FIELDS:
    raise ValueError('Notebook 01/02 row counts or features disagree.')
if a1['source_sha256'] != a2['source_sha256_from_notebook_01']:
    raise ValueError('Source files changed between Notebook 01 and Notebook 02.')
print('Dataset: ToN-IoT | Records:', EXPECTED_ROWS, '| Features:', len(FEATURES))
print('All rows kept. Excel copies will be saved next to each CSV.GZ file.')

## 1. Load and verify Core10 data

Check labels, row IDs and columns before splitting.

In [ ]:
frame = pd.read_csv(SELECTED, dtype=str, keep_default_na=False, low_memory=False)
if frame.columns.tolist() != FIELDS:
    raise ValueError(f'Wrong exported columns: {frame.columns.tolist()}')
if len(frame) != EXPECTED_ROWS:
    raise ValueError(f'Expected {EXPECTED_ROWS:,} rows, got {len(frame):,}')
ids = pd.to_numeric(frame['csv_row_index_0based'], errors='raise').to_numpy(dtype=np.int64)
if not np.array_equal(ids, np.arange(EXPECTED_ROWS)):
    raise ValueError('Source row IDs must be complete and in original order.')
if set(frame['label']) != {'0', '1'}:
    raise ValueError('Expected 0=Normal and 1=Attack labels.')
if int(((frame['label'] == '0') != frame['type'].str.casefold().eq('normal')).sum()):
    raise ValueError('The binary label and attack-type label disagree.')
print('Verified:', f'{len(frame):,}', 'rows; 13 columns; all original row IDs present.')
display(frame['label'].map({'0':'Normal', '1':'Attack'}).value_counts().rename('records').to_frame())

## 2. Group identical feature patterns

Use the same normalization as Notebook 02. Keep each repeated feature group in one partition.

In [ ]:
canonical = pd.DataFrame(index=frame.index)
for feature in FEATURES:
    raw = frame[feature].astype('string').str.strip()
    absent = raw.str.casefold().isin(MISSING).fillna(True)
    if feature in NUMERIC:
        values = pd.to_numeric(raw.mask(absent), errors='coerce')
        canonical[feature] = values.astype('string').fillna('__MISSING__')
    else:
        canonical[feature] = raw.str.casefold().mask(absent, '__MISSING__').fillna('__MISSING__')

# Same pandas 64-bit fingerprint method as Notebook 02.
fingerprints = pd.util.hash_pandas_object(canonical, index=False).to_numpy(dtype=np.uint64)
group_source = pd.DataFrame({
    'signature': fingerprints,
    'attack': frame['label'].eq('1').to_numpy(dtype=np.int8)
})
groups = group_source.groupby('signature', sort=False).agg(
    rows=('attack', 'size'), attacks=('attack', 'sum')
)
groups['normal'] = groups['rows'] - groups['attacks']
previous = a2['feature_pattern_diagnostics']
if len(groups) != previous['distinct_normalized_feature_patterns']:
    raise ValueError('Normalized pattern counts differ from Notebook 02.')
print(f'Feature groups: {len(groups):,}; repeated: {(groups.rows > 1).sum():,}')
print('Feature fingerprints are a grouping method, not proof of independent sessions.')

## 3. Create or reuse the frozen split

Balance Normal/Attack records as closely as possible while keeping feature groups together. Reuse an existing manifest without changing it.

In [ ]:
manifest_columns = ['csv_row_index_0based', 'partition', 'label', 'core10_signature_hex']
signature_hex = np.fromiter((f'{int(x):016x}' for x in fingerprints), dtype='<U16', count=len(frame))
if MANIFEST.is_file():
    saved = pd.read_csv(MANIFEST, dtype=str, keep_default_na=False)
    if saved.columns.tolist() != manifest_columns or len(saved) != EXPECTED_ROWS:
        raise ValueError('Existing frozen manifest schema or row count is wrong.')
    saved_ids = pd.to_numeric(saved['csv_row_index_0based'], errors='raise').to_numpy(dtype=np.int64)
    if (not np.array_equal(saved_ids, ids) or
        not np.array_equal(saved['label'].to_numpy(), frame['label'].to_numpy()) or
        not np.array_equal(saved['core10_signature_hex'].to_numpy(), signature_hex)):
        raise ValueError('Frozen manifest does not match the current Core10 dataset. Do not re-split.')
    partition = saved['partition'].to_numpy()
    if not set(np.unique(partition)).issubset(set(PARTS)):
        raise ValueError('Unknown partition value in saved manifest.')
    print('Reusing existing frozen manifest:', MANIFEST.name)
else:
    if any(SPLITS.glob('ton_iot_core10_*_v3.csv.gz')):
        raise FileExistsError('Split files exist but the frozen manifest is missing. Review files before continuing.')
    sizes = groups['rows'].to_numpy(dtype=np.int64)
    normal = groups['normal'].to_numpy(dtype=np.int64)
    attacks = groups['attacks'].to_numpy(dtype=np.int64)
    total_normal, total_attack = int(normal.sum()), int(attacks.sum())
    quotas = FRACTIONS[:, None] * np.array([total_normal, total_attack])[None, :]
    current = np.zeros((3, 2), dtype=float)
    rng = np.random.default_rng(SEED)
    order = np.lexsort((rng.random(len(sizes)), -sizes))
    chosen = np.empty(len(sizes), dtype=np.uint8)
    scale = np.array([max(total_normal, 1)**2, max(total_attack, 1)**2], dtype=float)
    for group_index in order:
        counts = np.array([normal[group_index], attacks[group_index]], dtype=float)
        # Greedy improvement toward per-class 70/15/15 targets.
        gains = ((2 * (quotas - current) * counts - counts * counts) / scale).sum(axis=1)
        choice = int(np.argmax(gains))
        chosen[group_index] = choice
        current[choice] += counts
    group_map = pd.Series(chosen, index=groups.index)
    record_assignment = pd.Series(fingerprints).map(group_map)
    if record_assignment.isna().any():
        raise ValueError('Some feature groups were not assigned.')
    partition = np.array(PARTS)[record_assignment.to_numpy(dtype=np.int64)]
    new_manifest = pd.DataFrame({
        'csv_row_index_0based': ids,
        'partition': partition,
        'label': frame['label'].to_numpy(),
        'core10_signature_hex': signature_hex
    })
    tmp_manifest = Path(str(MANIFEST) + '.tmp')
    if tmp_manifest.exists():
        raise FileExistsError(f'Partial manifest exists: {tmp_manifest}')
    # Verify group separation and approximate targets BEFORE freezing.
    check = pd.DataFrame({'signature': fingerprints, 'partition': partition})
    overlaps = int((check.groupby('signature')['partition'].nunique() > 1).sum())
    ratios = np.array([(partition == p).sum() for p in PARTS]) / len(partition)
    if overlaps or np.any(abs(ratios - FRACTIONS) > .03):
        raise ValueError(f'Cannot safely freeze group-disjoint split: overlap={overlaps}, ratios={ratios}')
    new_manifest.to_csv(tmp_manifest, index=False)
    tmp_manifest.replace(MANIFEST)
    print('Created frozen split manifest:', MANIFEST.name)

check = pd.DataFrame({'signature': fingerprints, 'partition': partition})
overlap_groups = int((check.groupby('signature')['partition'].nunique() > 1).sum())
if overlap_groups:
    raise ValueError(f'{overlap_groups} identical-feature groups cross partitions. STOP.')
rows = []
for p in PARTS:
    view = frame.loc[partition == p, 'label']
    norm = int((view == '0').sum()); atk = int((view == '1').sum())
    rows.append({'partition': p, 'Normal': norm, 'Attack': atk, 'Total': len(view)})
summary = pd.DataFrame(rows).set_index('partition')
if summary['Total'].sum() != EXPECTED_ROWS or not ((summary['Normal'] > 0) & (summary['Attack'] > 0)).all():
    raise ValueError('Invalid partition coverage or missing class.')
ratios = summary['Total'].to_numpy() / EXPECTED_ROWS
if np.any(abs(ratios - FRACTIONS) > .03):
    raise ValueError(f'Frozen split is too far from 70/15/15: {ratios}')
print(f'All records preserved: {summary.Total.sum():,}/{EXPECTED_ROWS:,} (100%)')
print('Overlapping Core10 fingerprint groups:', overlap_groups)
display(summary)

## 4. Save and verify split CSV.GZ files

Keep the original record order and labels. Never silently overwrite an existing frozen split.

In [ ]:
OUTPUTS = {p: SPLITS / f'ton_iot_core10_{p}_v3.csv.gz' for p in PARTS}
file_details = {}
for p in PARTS:
    dest = OUTPUTS[p]
    partition_rows = frame.loc[partition == p, FIELDS]
    tmp = Path(str(dest) + '.tmp')
    if tmp.exists():
        raise FileExistsError(f'Partial split exists: {tmp}. Inspect before rerunning.')
    if not dest.exists():
        partition_rows.to_csv(tmp, index=False, compression='gzip')
        # Output is verified below; keep original frozen partition data protected.
        tmp.replace(dest)
        print('Created:', dest.name)
    else:
        print('Reusing:', dest.name)
    saved = pd.read_csv(dest, dtype=str, keep_default_na=False, low_memory=False)
    if saved.columns.tolist() != FIELDS or not saved.equals(partition_rows.reset_index(drop=True)):
        raise ValueError(f'{p}: existing CSV.GZ differs from frozen partition; DO NOT overwrite.')
    file_details[p] = {'name': dest.name, 'rows': int(len(saved)), 'bytes': dest.stat().st_size}
    del saved

if sum(x['rows'] for x in file_details.values()) != EXPECTED_ROWS:
    raise ValueError('Export lost source records.')
print('Verified all three full CSV.GZ datasets. No rows removed.')

## 5. Save split report and graph

Report exact partition sizes. Only training and validation will be used in model development.

In [ ]:
ax = summary[['Normal','Attack']].plot(kind='bar', stacked=True, figsize=(8, 4.4), rot=0)
ax.set_title('ToN-IoT — Frozen train / validation / test split')
ax.set_ylabel('Records')
ax.grid(axis='y', alpha=.2)
plt.tight_layout()
plt.savefig(FIGURE, dpi=180, bbox_inches='tight')
plt.show()
plt.close()

sha = hashlib.sha256()
with MANIFEST.open('rb') as handle:
    for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
        sha.update(block)
report = {
    'dataset': 'ToN-IoT', 'source_rows': EXPECTED_ROWS,
    'model_features': FEATURES, 'categorical_features': CATEGORICAL,
    'target': 'label', 'attack_type': 'type',
    'manifest_name': MANIFEST.name, 'manifest_sha256': sha.hexdigest(),
    'seed_for_first_split': SEED,
    'method': 'class-aware greedy assignment of entire normalized Core10 feature-fingerprint groups',
    'requested_fractions': dict(zip(PARTS, FRACTIONS.tolist())),
    'counts': {p: {'rows': int(summary.at[p,'Total']), 'normal': int(summary.at[p,'Normal']), 'attack': int(summary.at[p,'Attack'])} for p in PARTS},
    'total_rows_preserved': int(summary.Total.sum()),
    'normalized_fingerprint_overlap_groups': overlap_groups,
    'num_columns': len(FIELDS), 'exported_columns': FIELDS,
    'files': file_details, 'test_scored': False,
    'limitations': [
        'Feature fingerprint separation does not prove independent capture sessions or attack campaigns.',
        'The dataset is class-imbalanced; use precision, recall and false-positive rate, not only accuracy.',
        'The service field has many placeholders and may create a dataset shortcut.',
        'These are ToN-IoT-specific features, not verified transferable X-IIoTID inputs.'
    ]
}
REPORT.write_text(json.dumps(report, indent=2), encoding='utf-8')
print('Saved report:', REPORT)
print('Test data has not been scored.')

## 6. Export Excel beside each CSV.GZ

Create three readable `.xlsx` files in the **same** `data/splits/ton_iot/` folder. Excel is for inspection, not ML training.

In [ ]:
if EXPORT_EXCEL:
    if not EXCEL_SCRIPT.is_file():
        raise FileNotFoundError(f'Excel export script missing: {EXCEL_SCRIPT}')
    subprocess.run([
        'powershell.exe', '-NoProfile', '-ExecutionPolicy', 'Bypass',
        '-File', str(EXCEL_SCRIPT), '-ProjectRoot', str(ROOT)
    ], check=True)
    for p in PARTS:
        xlsx = OUTPUTS[p].with_suffix('').with_suffix('.xlsx')
        if not xlsx.is_file() or xlsx.stat().st_size == 0:
            raise FileNotFoundError(f'Excel export failed: {xlsx}')
    print('All three full Excel files are beside their CSV.GZ sources.')
else:
    print('Excel export disabled. CSV.GZ data remains safe; set EXPORT_EXCEL=True when Excel is available.')
print('Next: 04_model_training.ipynb — train only on train; validate on validation; keep test sealed.')